# Brazilian E-Commerce Analysis — Olist Dataset
## Phase 3: SQL Business Analysis
**Author:** Yasser Ramzy  
**Database:** MySQL — olist_ecommerce  
**Tools:** Python · mysql-connector-python · pandas  

### Business Questions We Will Answer
1. Which states generate the most revenue?
2. Who are the top performing sellers?
3. How does late delivery impact revenue and reviews?
4. What is the RFM segmentation of customers?
5. What are the peak shopping hours?

In [1]:
import pandas as pd
import mysql.connector
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

# ── Connect to MySQL ─────────────────────────────────────────
conn = mysql.connector.connect(
    host     = 'localhost',
    user     = 'root',
    password = 'yasserkamelramzy',  # ← replace with your password
    database = 'olist_ecommerce'
)

cursor = conn.cursor()
print("✅ Connected to MySQL — olist_ecommerce")

✅ Connected to MySQL — olist_ecommerce


## Step 1: Load Clean Data into MySQL
We create the master table in MySQL and load our clean dataset from Phase 1.

In [2]:
# ── Load clean CSV ───────────────────────────────────────────
path = r"C:\Users\yasse\OneDrive\Desktop\data analysis projects\Brazilian-E-Commerce-SQL-POWERBI\Brazilian-Olist-SQL-POWERBI-finished-project\outputs\master_delivered.csv"

df = pd.read_csv(path, parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

print(f"✅ CSV loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ── Create table in MySQL ────────────────────────────────────
cursor.execute("DROP TABLE IF EXISTS master_orders;")

create_table = """
CREATE TABLE master_orders (
    order_id                        VARCHAR(50),
    customer_id                     VARCHAR(50),
    order_status                    VARCHAR(20),
    order_purchase_timestamp        DATETIME,
    order_approved_at               DATETIME,
    order_delivered_carrier_date    DATETIME,
    order_delivered_customer_date   DATETIME,
    order_estimated_delivery_date   DATETIME,
    customer_unique_id              VARCHAR(50),
    customer_zip_code_prefix        INT,
    customer_city                   VARCHAR(100),
    customer_state                  VARCHAR(5),
    order_total_price               FLOAT,
    order_freight_value             FLOAT,
    order_items_count               INT,
    product_id                      VARCHAR(50),
    seller_id                       VARCHAR(50),
    product_category_name_english   VARCHAR(100),
    product_weight_g                FLOAT,
    product_length_cm               FLOAT,
    product_height_cm               FLOAT,
    product_width_cm                FLOAT,
    seller_city                     VARCHAR(100),
    seller_state                    VARCHAR(5),
    total_payment_value             FLOAT,
    payment_installments            INT,
    payment_type                    VARCHAR(30),
    review_score                    FLOAT,
    review_comment_message          TEXT,
    delivery_days                   FLOAT,
    delivery_delay_days             FLOAT,
    is_late                         INT,
    order_total_value               FLOAT,
    purchase_year                   INT,
    purchase_month                  INT,
    purchase_month_name             VARCHAR(5),
    purchase_day_of_week            VARCHAR(15),
    freight_ratio                   FLOAT
);
"""
cursor.execute(create_table)
conn.commit()
print("✅ Table created in MySQL")

✅ CSV loaded: 96,469 rows × 39 columns
✅ Table created in MySQL


In [3]:
# ── Insert data in batches ───────────────────────────────────
# Replace NaN with None so MySQL accepts NULL values
df = df.where(pd.notnull(df), None)

# Drop review_creation_date column if it exists
if 'review_creation_date' in df.columns:
    df = df.drop(columns=['review_creation_date'])

cols = list(df.columns)
placeholders = ', '.join(['%s'] * len(cols))
insert_sql = f"INSERT INTO master_orders ({', '.join(cols)}) VALUES ({placeholders})"

# Insert in batches of 5000 rows
batch_size = 5000
total      = len(df)
inserted   = 0

for start in range(0, total, batch_size):
    batch = df.iloc[start:start + batch_size]
    data  = [tuple(row) for row in batch.itertuples(index=False)]
    cursor.executemany(insert_sql, data)
    conn.commit()
    inserted += len(batch)
    print(f"   Inserted {inserted:,} / {total:,} rows...", end='\r')

print(f"\n✅ All {inserted:,} rows loaded into MySQL successfully!")

# Verify
cursor.execute("SELECT COUNT(*) FROM master_orders;")
count = cursor.fetchone()[0]
print(f"✅ Verified: {count:,} rows in master_orders table")

   Inserted 96,469 / 96,469 rows...
✅ All 96,469 rows loaded into MySQL successfully!
✅ Verified: 96,469 rows in master_orders table


## Step 2: Business Analysis Queries

### Query 1: Revenue & Orders by State
Which Brazilian states generate the most revenue?

In [4]:
# ── Query 1: Revenue by state ────────────────────────────────
q1 = """
SELECT 
    customer_state,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct
FROM master_orders
GROUP BY customer_state
ORDER BY total_revenue DESC;
"""

df_q1 = pd.read_sql(q1, conn)
print("── Revenue by State ────────────────────────────────")
print(df_q1.to_string(index=False))

# Save to sql folder
sql_path = r"C:\Users\yasse\OneDrive\Desktop\data analysis projects\Brazilian-E-Commerce-SQL-POWERBI\Brazilian-Olist-SQL-POWERBI-finished-project\SQL"

df_q1.to_csv(os.path.join(sql_path, 'q1_revenue_by_state.csv'), index=False)
print("\n✅ Query 1 saved")

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\368656161.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q1 = pd.read_sql(q1, conn)


── Revenue by State ────────────────────────────────
customer_state  total_orders  total_revenue  avg_order_value  avg_review  avg_delivery_days  late_rate_pct
            SP         40493     5768374.77           142.45        4.22                8.3            4.5
            RJ         12350     2055401.57           166.43        3.92               14.8           12.1
            MG         11354     1818891.67           160.20        4.17               11.5            4.6
            RS          5344      861278.79           161.17        4.17               14.8            6.1
            PR          4923      781708.80           158.79        4.22               11.5            4.0
            SC          3546      595127.78           167.83        4.10               14.5            8.2
            BA          3256      591137.81           181.55        3.90               18.9           12.2
            DF          2080      346123.35           166.41        4.11               12.5

### Query 2: Top Performing Sellers
Who are the top 20 sellers by revenue, and how do they perform on reviews and delivery?

In [5]:
# ── Query 2: Top sellers ─────────────────────────────────────
q2 = """
SELECT 
    seller_id,
    seller_state,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct,
    COUNT(DISTINCT customer_state)      AS states_served
FROM master_orders
GROUP BY seller_id, seller_state
HAVING total_orders >= 50
ORDER BY total_revenue DESC
LIMIT 20;
"""

df_q2 = pd.read_sql(q2, conn)
df_q2['seller_id'] = df_q2['seller_id'].str[:8] + '...'
print("── Top 20 Sellers ──────────────────────────────────")
print(df_q2.to_string(index=False))

df_q2.to_csv(os.path.join(sql_path, 'q2_top_sellers.csv'), index=False)
print("\n✅ Query 2 saved")

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\3929275581.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q2 = pd.read_sql(q2, conn)


── Top 20 Sellers ──────────────────────────────────
  seller_id seller_state  total_orders  total_revenue  avg_order_value  avg_review  avg_delivery_days  late_rate_pct  states_served
4869f7a5...           SP          1117      247937.80           221.97        4.13               14.6           10.6             27
7c67e144...           SP           966      235936.88           244.24        3.48               22.0            9.2             24
4a3ca931...           SP          1726      233449.18           135.25        3.84               14.1           10.0             25
53243585...           BA           348      230797.02           663.21        4.17               13.0            3.5             26
fa1c13f2...           SP           571      200044.11           350.34        4.36               12.9            9.3             24
da8622b1...           SP          1295      185792.71           143.47        4.17               10.8            6.7             23
1025f0e2...           S

### Query 3: Late Delivery Impact Analysis
How does late delivery affect revenue, review scores, and payment behavior?

In [6]:
# ── Query 3: Late delivery impact ───────────────────────────
q3 = """
SELECT
    is_late,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review_score,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(payment_installments), 1) AS avg_installments,
    ROUND(AVG(freight_ratio) * 100, 1)  AS avg_freight_pct
FROM master_orders
WHERE review_score > 0
GROUP BY is_late
ORDER BY is_late;
"""

df_q3 = pd.read_sql(q3, conn)
df_q3['is_late'] = df_q3['is_late'].map({0: 'On Time', 1: 'Late'})
print("── Late Delivery Impact ────────────────────────────")
print(df_q3.to_string(index=False))

df_q3.to_csv(os.path.join(sql_path, 'q3_late_delivery_impact.csv'), index=False)
print("\n✅ Query 3 saved")

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\3226802075.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q3 = pd.read_sql(q3, conn)


── Late Delivery Impact ────────────────────────────
is_late  total_orders  total_revenue  avg_order_value  avg_review_score  avg_delivery_days  avg_installments  avg_freight_pct
On Time         89443    14176106.46           158.49              4.29               10.5               2.9             20.9
   Late          6380     1113724.47           174.56              2.27               33.4               3.0             21.3

✅ Query 3 saved


### Query 4: RFM Customer Segmentation
We calculate Recency, Frequency, and Monetary scores for every customer
to segment them into actionable groups.

In [7]:
# ── Query 4: RFM Segmentation ────────────────────────────────
q4 = """
WITH rfm_base AS (
    SELECT
        customer_unique_id,
        DATEDIFF('2018-09-01', MAX(order_purchase_timestamp)) AS recency,
        COUNT(DISTINCT order_id)                               AS frequency,
        ROUND(SUM(order_total_value), 2)                       AS monetary
    FROM master_orders
    GROUP BY customer_unique_id
),
rfm_scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency DESC)   AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC)  AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC)   AS m_score
    FROM rfm_base
),
rfm_segmented AS (
    SELECT *,
        CONCAT(r_score, f_score, m_score) AS rfm_score,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4
                THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3
                THEN 'Loyal Customers'
            WHEN r_score >= 4 AND f_score <= 2
                THEN 'New Customers'
            WHEN r_score <= 2 AND f_score >= 3
                THEN 'At Risk'
            WHEN r_score <= 2 AND f_score <= 2 AND m_score <= 2
                THEN 'Lost'
            ELSE 'Potential Loyalists'
        END AS segment
    FROM rfm_scored
)
SELECT
    segment,
    COUNT(*)                        AS customers,
    ROUND(AVG(recency), 0)          AS avg_recency_days,
    ROUND(AVG(frequency), 2)        AS avg_frequency,
    ROUND(AVG(monetary), 2)         AS avg_monetary,
    ROUND(SUM(monetary), 2)         AS total_revenue
FROM rfm_segmented
GROUP BY segment
ORDER BY total_revenue DESC;
"""

df_q4 = pd.read_sql(q4, conn)
print("── RFM Customer Segments ───────────────────────────")
print(df_q4.to_string(index=False))

df_q4.to_csv(os.path.join(sql_path, 'q4_rfm_segments.csv'), index=False)
print("\n✅ Query 4 saved")

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\2537604122.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q4 = pd.read_sql(q4, conn)


── RFM Customer Segments ───────────────────────────
            segment  customers  avg_recency_days  avg_frequency  avg_monetary  total_revenue
            At Risk      37340             397.0           1.03        165.33     6173487.03
      New Customers      30015              79.0           1.00        164.53     4938294.36
    Loyal Customers      17685             207.0           1.05        159.97     2829001.69
Potential Loyalists       7325             205.0           1.00        151.63     1110663.12
          Champions        984              91.0           2.18        372.77      366805.17

✅ Query 4 saved


### Query 5: Peak Shopping Hours
When do Brazilian customers place their orders throughout the day?

In [8]:
# ── Query 5: Peak shopping hours ────────────────────────────
q5 = """
SELECT
    HOUR(order_purchase_timestamp)      AS hour_of_day,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review
FROM master_orders
GROUP BY hour_of_day
ORDER BY hour_of_day;
"""

df_q5 = pd.read_sql(q5, conn)
print("── Orders by Hour of Day ───────────────────────────")
for _, row in df_q5.iterrows():
    bar = '█' * int(row['total_orders'] / 200)
    print(f"  {int(row['hour_of_day']):02d}:00  {bar:<35} {int(row['total_orders']):,}")

df_q5.to_csv(os.path.join(sql_path, 'q5_peak_hours.csv'), index=False)
print("\n✅ Query 5 saved")

── Orders by Hour of Day ───────────────────────────
  00:00  ███████████                         2,321
  01:00  █████                               1,133
  02:00  ██                                  496
  03:00  █                                   259
  04:00  █                                   203
  05:00                                      182
  06:00  ██                                  477
  07:00  █████                               1,199
  08:00  ██████████████                      2,907
  09:00  ███████████████████████             4,647
  10:00  █████████████████████████████       5,978
  11:00  ███████████████████████████████     6,385
  12:00  █████████████████████████████       5,801
  13:00  ███████████████████████████████     6,309
  14:00  ███████████████████████████████     6,383
  15:00  ███████████████████████████████     6,249
  16:00  ████████████████████████████████    6,475
  17:00  █████████████████████████████       5,961
  18:00  ███████████████████████████   

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\3447683580.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q5 = pd.read_sql(q5, conn)


### Query 6: Payment Behavior Analysis
How do customers pay, and does payment type affect order value or satisfaction?

In [9]:
# ── Query 6: Payment behavior ────────────────────────────────
q6 = """
SELECT
    payment_type,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(payment_installments), 1) AS avg_installments,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct
FROM master_orders
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_revenue DESC;
"""

df_q6 = pd.read_sql(q6, conn)
print("── Payment Behavior ────────────────────────────────")
print(df_q6.to_string(index=False))

df_q6.to_csv(os.path.join(sql_path, 'q6_payment_behavior.csv'), index=False)
print("\n✅ Query 6 saved")

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\2302880953.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q6 = pd.read_sql(q6, conn)


── Payment Behavior ────────────────────────────────
payment_type  total_orders  total_revenue  avg_order_value  avg_installments  avg_review  late_rate_pct
 credit_card         73934    12228846.96           165.40               3.5        4.13            6.7
      boleto         19191     2769932.71           144.33               1.0        4.13            7.3
     voucher          1861      211402.83           113.60               1.1        4.06            5.9
  debit_card          1483      208068.87           140.30               1.0        4.22            5.3

✅ Query 6 saved


### Query 7: Category Performance Deep Dive
Which categories have the best combination of revenue, review score, and delivery speed?

In [10]:
# ── Query 7: Category deep dive ──────────────────────────────
q7 = """
SELECT
    product_category_name_english,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct,
    ROUND(AVG(freight_ratio) * 100, 1)  AS avg_freight_pct
FROM master_orders
WHERE product_category_name_english != 'unknown'
GROUP BY product_category_name_english
HAVING total_orders >= 100
ORDER BY total_revenue DESC
LIMIT 20;
"""

df_q7 = pd.read_sql(q7, conn)
print("── Category Performance ────────────────────────────")
print(df_q7.to_string(index=False))

df_q7.to_csv(os.path.join(sql_path, 'q7_category_performance.csv'), index=False)
print("\n✅ Query 7 saved")

C:\Users\yasse\AppData\Local\Temp\ipykernel_22520\2928273425.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_q7 = pd.read_sql(q7, conn)


── Category Performance ────────────────────────────
product_category_name_english  total_orders  total_revenue  avg_order_value  avg_review  avg_delivery_days  late_rate_pct  avg_freight_pct
                health_beauty          8607     1410754.41           163.91        4.21               11.6            7.5             19.4
                watches_gifts          5470     1261539.27           230.63        4.10               12.4            7.4             12.8
               bed_bath_table          9167     1223852.92           133.51        3.97               12.6            7.5             19.9
               sports_leisure          7490     1119060.16           149.41        4.21               11.8            6.6             19.9
        computers_accessories          6500     1030615.59           158.56        4.06               12.7            6.4             20.6
              furniture_decor          6213      884057.80           142.29        4.04               12.6       

In [11]:
# ── Close MySQL connection ───────────────────────────────────
cursor.close()
conn.close()
print("✅ MySQL connection closed")
print("\n── Phase 3 Complete ────────────────────────────────")
print("   7 business queries written and saved to /sql/")
print("\n   Files saved:")
files = ['q1_revenue_by_state.csv', 'q2_top_sellers.csv',
         'q3_late_delivery_impact.csv', 'q4_rfm_segments.csv',
         'q5_peak_hours.csv', 'q6_payment_behavior.csv',
         'q7_category_performance.csv']
for f in files:
    print(f"   • {f}")

✅ MySQL connection closed

── Phase 3 Complete ────────────────────────────────
   7 business queries written and saved to /sql/

   Files saved:
   • q1_revenue_by_state.csv
   • q2_top_sellers.csv
   • q3_late_delivery_impact.csv
   • q4_rfm_segments.csv
   • q5_peak_hours.csv
   • q6_payment_behavior.csv
   • q7_category_performance.csv


## Step 3: Save SQL Query Files
We save each business query as a standalone .sql file for the GitHub repository.

In [12]:
# ── Save all queries as .sql files ──────────────────────────
queries = {
    '01_revenue_by_state.sql': """
-- ============================================================
-- Query 1: Revenue & Performance by State
-- Author : Yasser Ramzy
-- Purpose: Identify which Brazilian states generate the most
--          revenue and how delivery performance varies by region
-- ============================================================

SELECT 
    customer_state,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct
FROM master_orders
GROUP BY customer_state
ORDER BY total_revenue DESC;
""",

    '02_top_sellers.sql': """
-- ============================================================
-- Query 2: Top Performing Sellers
-- Author : Yasser Ramzy
-- Purpose: Rank sellers by revenue and evaluate their delivery
--          and satisfaction performance (min 50 orders)
-- ============================================================

SELECT 
    seller_id,
    seller_state,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct,
    COUNT(DISTINCT customer_state)      AS states_served
FROM master_orders
GROUP BY seller_id, seller_state
HAVING total_orders >= 50
ORDER BY total_revenue DESC
LIMIT 20;
""",

    '03_late_delivery_impact.sql': """
-- ============================================================
-- Query 3: Late Delivery Impact Analysis
-- Author : Yasser Ramzy
-- Purpose: Measure how late delivery affects revenue,
--          review scores, and payment behavior
-- ============================================================

SELECT
    is_late,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review_score,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(payment_installments), 1) AS avg_installments,
    ROUND(AVG(freight_ratio) * 100, 1)  AS avg_freight_pct
FROM master_orders
WHERE review_score > 0
GROUP BY is_late
ORDER BY is_late;
""",

    '04_rfm_segmentation.sql': """
-- ============================================================
-- Query 4: RFM Customer Segmentation
-- Author : Yasser Ramzy
-- Purpose: Segment customers into actionable groups based on
--          Recency, Frequency, and Monetary scores
-- ============================================================

WITH rfm_base AS (
    SELECT
        customer_unique_id,
        DATEDIFF('2018-09-01', MAX(order_purchase_timestamp)) AS recency,
        COUNT(DISTINCT order_id)                               AS frequency,
        ROUND(SUM(order_total_value), 2)                       AS monetary
    FROM master_orders
    GROUP BY customer_unique_id
),
rfm_scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency DESC)   AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC)  AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC)   AS m_score
    FROM rfm_base
),
rfm_segmented AS (
    SELECT *,
        CONCAT(r_score, f_score, m_score) AS rfm_score,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4
                THEN 'Champions'
            WHEN r_score >= 3 AND f_score >= 3
                THEN 'Loyal Customers'
            WHEN r_score >= 4 AND f_score <= 2
                THEN 'New Customers'
            WHEN r_score <= 2 AND f_score >= 3
                THEN 'At Risk'
            WHEN r_score <= 2 AND f_score <= 2 AND m_score <= 2
                THEN 'Lost'
            ELSE 'Potential Loyalists'
        END AS segment
    FROM rfm_scored
)
SELECT
    segment,
    COUNT(*)                        AS customers,
    ROUND(AVG(recency), 0)          AS avg_recency_days,
    ROUND(AVG(frequency), 2)        AS avg_frequency,
    ROUND(AVG(monetary), 2)         AS avg_monetary,
    ROUND(SUM(monetary), 2)         AS total_revenue
FROM rfm_segmented
GROUP BY segment
ORDER BY total_revenue DESC;
""",

    '05_peak_hours.sql': """
-- ============================================================
-- Query 5: Peak Shopping Hours
-- Author : Yasser Ramzy
-- Purpose: Identify when customers place orders throughout
--          the day to optimize marketing and operations
-- ============================================================

SELECT
    HOUR(order_purchase_timestamp)      AS hour_of_day,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review
FROM master_orders
GROUP BY hour_of_day
ORDER BY hour_of_day;
""",

    '06_payment_behavior.sql': """
-- ============================================================
-- Query 6: Payment Behavior Analysis
-- Author : Yasser Ramzy
-- Purpose: Understand how payment type affects order value,
--          installment usage, and customer satisfaction
-- ============================================================

SELECT
    payment_type,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(payment_installments), 1) AS avg_installments,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct
FROM master_orders
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_revenue DESC;
""",

    '07_category_performance.sql': """
-- ============================================================
-- Query 7: Category Performance Deep Dive
-- Author : Yasser Ramzy
-- Purpose: Rank product categories by revenue and evaluate
--          their review scores, delivery speed, and freight cost
-- ============================================================

SELECT
    product_category_name_english,
    COUNT(DISTINCT order_id)            AS total_orders,
    ROUND(SUM(order_total_value), 2)    AS total_revenue,
    ROUND(AVG(order_total_value), 2)    AS avg_order_value,
    ROUND(AVG(review_score), 2)         AS avg_review,
    ROUND(AVG(delivery_days), 1)        AS avg_delivery_days,
    ROUND(AVG(is_late) * 100, 1)        AS late_rate_pct,
    ROUND(AVG(freight_ratio) * 100, 1)  AS avg_freight_pct
FROM master_orders
WHERE product_category_name_english != 'unknown'
GROUP BY product_category_name_english
HAVING total_orders >= 100
ORDER BY total_revenue DESC
LIMIT 20;
"""
}

# Save each query as a .sql file
for filename, query in queries.items():
    filepath = os.path.join(sql_path, filename)
    with open(filepath, 'w') as f:
        f.write(query.strip())
    print(f"   ✅ Saved: {filename}")

print(f"\n🎉 All 7 SQL files saved to /sql/ folder")

   ✅ Saved: 01_revenue_by_state.sql
   ✅ Saved: 02_top_sellers.sql
   ✅ Saved: 03_late_delivery_impact.sql
   ✅ Saved: 04_rfm_segmentation.sql
   ✅ Saved: 05_peak_hours.sql
   ✅ Saved: 06_payment_behavior.sql
   ✅ Saved: 07_category_performance.sql

🎉 All 7 SQL files saved to /sql/ folder
